# Análise Exploratória de Dados

- Equipe 5: `Hakuna Ma-Data`
- Conjunto de Dados `spanish_wines`(Spanish Wines)


- Cientistas de Dados:
  - Danilo Dos Santos Sampaio Gonçalves (danilosampaio4747@gmail.com)
  - Jaime Teixeira De Araújo Júnior (jaimetjribeiro@gmail.com)
---


In [4]:
# @title Carregando Bibliotecas
import pandas as pd
import os
import numpy as np
import seaborn as sns
import itertools
import kagglehub
from matplotlib import pyplot as plt
from IPython.display import Markdown


In [7]:
path = kagglehub.dataset_download("fedesoriano/spanish-wine-quality-dataset")

print("Path to dataset files:", path)

print("Nome do arquivo:", os.listdir(path))


Path to dataset files: C:\Users\jaime\.cache\kagglehub\datasets\fedesoriano\spanish-wine-quality-dataset\versions\1
Nome do arquivo: ['wines_SPA.csv']


In [ ]:

df = pd.read_csv(os.path.join(path, 'wines_SPA.csv'))
df.head()

In [ ]:
# Tipo de dados das variáveis
df.dtypes

In [ ]:
# Definindo tema do seaborn
sns.set_theme(style="whitegrid")

### Visualização dos dados

In [ ]:
display(Markdown("### Primeiras linhas"))
df.head()

In [ ]:
display(Markdown("### Últimas linhas"))
df.tail()

In [ ]:
display(Markdown("### Informações das variáveis"))
df.info()

- Unidades amostrais: 7500
- Quantidade de variáveis: 11

In [ ]:
display(Markdown("### Valores únicos"))
df.nunique()

Como a coluna **year** foi identificada como tipo object, foi feita a observação dos valores únicos para entender quais são não numéricos.

In [ ]:
# Valores únicos da coluna "year", que é do tipo object
print(df['year'].unique())

In [ ]:
# @title Dicionário de dados
df_dict = pd.DataFrame([
    {
        "variavel": "winery",
        "descricao": "Nome da vinícola.",
        "tipo": "qualitativa",
        "subtipo": "nominal",
    },
    {
        "variavel": "wine",
        "descricao": "Nome do vinho.",
        "tipo": "qualitativa",
        "subtipo": "nominal",
    },
    {
        "variavel": "year",
        "descricao": "Ano da safra do vinho.",
        "tipo": "qualitativa",
        "subtipo": "ordinal",
    },
    {
        "variavel": "rating",
        "descricao": "Avaliação média dos usuários.",
        "tipo": "quantitativa",
        "subtipo": "contínua",
    },
    {
        "variavel": "num_reviews",
        "descricao": "Quantidade de avaliações recebidas.",
        "tipo": "quantitativa",
        "subtipo": "discreta",
    },
    {
        "variavel": "country",
        "descricao": "País de origem.",
        "tipo": "qualitativa",
        "subtipo": "nominal",
    },
    {
        "variavel": "region",
        "descricao": "Região de produção.",
        "tipo": "qualitativa",
        "subtipo": "nominal",
    },
    {
        "variavel": "price",
        "descricao": "Preço em Euros.",
        "tipo": "quantitativa",
        "subtipo": "contínua",
    },
    {
        "variavel": "type",
        "descricao": "Tipo do vinho.",
        "tipo": "qualitativa",
        "subtipo": "nominal",
    },
    {
        "variavel": "body",
        "descricao": "Intensidade do corpo do vinho, escala 1-5.",
        "tipo": "quantitativa",
        "subtipo": "discreta",
    },
    {
        "variavel": "acidity",
        "descricao": "Nível de acidez, escala 1-5.",
        "tipo": "quantitativa",
        "subtipo": "discreta"
    }
])

df_dict

---
Obs: A presença do valor “N.V.” na variável **year** indica vinhos classificados como não vintage (sem safra definida), isso significa que o vinho não está associado a um único ano de colheita, pois pode ter sido produzido a partir da combinação de uvas de diferentes safras.

Por esse motivo, a coluna foi importada como texto e está sendo tratada **inicialmente** como qualitativa ordinal.

---

## Análise Univariada

In [ ]:
# @title Resumo Estatístico

display(Markdown("#### Variáveis Qualitativas"))
display(df.describe(include='object'))

display(Markdown("#### Variáveis Quantitativas"))
display(df.describe())

### Distribuição de Variáveis

In [ ]:
# @title Variáveis Quantitativas

variaveis_quantitativas = df_dict.query("tipo == 'quantitativa'").variavel.to_list()

fig, axes = plt.subplots(figsize=(18, 6), ncols=len(variaveis_quantitativas), nrows=2,
                         gridspec_kw={"height_ratios": [3, 1]})

for i, variavel in enumerate(variaveis_quantitativas):

    use_log = True if variavel in ['price', 'num_reviews'] else False


    ax = sns.histplot(data=df, x=variavel, ax=axes[0, i], kde=True, alpha=.8, log_scale=(use_log, False))


    mediana = df[variavel].median()
    media = df[variavel].mean()

    ax.axvline(mediana, color="blue", label="Mediana")
    ax.axvline(media, color="red", linestyle="--", label="Média")

    ax.set(title=f"Distribuição de {variavel}", ylabel="Quantidade", xlabel="")
    ax.legend()


    ax.spines["bottom"].set_color("black")
    ax.grid(False, axis="x")
    for side in ["left", "top", "right"]:
        ax.spines[side].set_visible(False)


    ax_box = sns.boxplot(data=df, x=variavel, ax=axes[1, i])
    if use_log:
        ax_box.set_xscale('log')
    ax_box.set(xlabel=f"Valores de {variavel}")

plt.tight_layout()
plt.show()

---

**Distribuição da variável rating**

- A maior parte dos vinhos está concentrada na nota 4.2, que coincide com a mediana e também é o valor mais frequente no conjunto.

- As notas acima de 4.2 aparecem de forma mais isolada, o que indica que avaliações muito altas existem, mas são exceções dentro do dataset.

**Distribuição da variável num_reviews**

- A maioria dos vinhos possui poucas avaliações, concentradas próximo ao zero.

- Existem vinhos com um volume de avaliações muito alto, considerados casos raros (outliers).

**Distribuição da variável price**

- A maioria dos preços está entre valores baixos (abaixo de 100€).

- Também fica evidente uma diferença considerável entre média e mediana, sugerindo que vinhos muito caros (acima de 500 euros) puxam a média para cima e acabam distorcendo a percepção do preço “típico”.

**Distribuição das variáveis body e acidity**

- A maior parte dos vinhos é classificada com corpo nível 4, mostrando um padrão bem concentrado nessa categoria.

- Vinhos com corpo mais leve (níveis 2 e 3) aparecem com bem menos frequência e ficam mais deslocados na distribuição.

**Distribuição da variável acidity**

- A acidez (acidity) apresenta menor variabilidade, concentrando-se no nível 3, funcionando como um padrão dentro do conjunto.

- Já os níveis mais baixos (1 e 2) são pouco frequentes e fogem da tendência predominante observada nos dados.

---

In [ ]:
# @title Variáveis Qualitativas

variaveis_qualitativas = df_dict.query("tipo == 'qualitativa'").variavel.to_list()

fig, axes = plt.subplots(figsize=(16, 18), ncols=2, nrows=3)
axes = axes.flatten()

for i, variavel in enumerate(variaveis_qualitativas):
    top_n = 15 if variavel == "year" else 10
    order = df[variavel].value_counts().index[:top_n]

    ax = sns.countplot(data=df, y=variavel, ax=axes[i], order=order, palette="Blues_d", hue=variavel, legend=False, alpha=.8)

    for bar in ax.patches:
        width = bar.get_width()

        if width > 80:
            ax.text(width/2, bar.get_y() + bar.get_height()/2,
                   f'{int(width)}',
                   ha='center', va='center', color='white', fontweight='bold', fontsize=10)
        else:
            ax.text(width + 15, bar.get_y() + bar.get_height()/2,
                   f'{int(width)}',
                   ha='left', va='center', color='black', fontsize=9)

    ax.set(title=f"Distribuição de {variavel} (Top {top_n})", xlabel="Quantidade")

    for side in ["bottom", "top", "right"]:
        ax.spines[side].set_visible(False)
    ax.spines["left"].set_color("black")

plt.tight_layout()
plt.show()

---

**Winery (Vinícola)**
 - A vinícola Contino é a que possui a maior quantidade de rótulos no dataset, destacando-se das demais que mantêm uma distribuição mais equilibrada entre 200 e 300 unidades.

**Wine (Vinho)**
-  Os nomes de vinhos mais frequentes são Reserva (467) e Gran Reserva (458).

**Year (Ano)**
- O ano de 2011 é a safra com maior presença absoluta, com quase 1.200 registros. É interessante notar a presença da categoria "N.V." entre os mais frequentes, representando vinhos sem uma safra específica definida.

**Country (País)**
- O dataset é composto exclusivamente por vinhos da Espanha, totalizando 7.500 registros.

**Region (Região)**
- A região de Rioja é a principal origem dos vinhos, com mais de 2.400 registros, seguida por Ribera del Duero. Juntas, elas dominam grande parte da amostra.

**Type (Tipo)**
- O tipo Rioja Red é o mais comum, o que é coerente com a dominância da região de Rioja observada anteriormente.

---

### Dados Faltantes

In [ ]:
df.isnull().sum()

---

Existem dados faltantes nas variáveis `year` (2), `type` (545), `body` (1169) e `acidity` (1169).

> **Estratégia adotada:** Os valores faltantes serão mantidos nos dados. As funções de visualização do Seaborn ignoram automaticamente registros com `NaN` nas variáveis utilizadas. Para a modelagem futura (notebook 02), será aplicado tratamento específico.

> **Nota sobre a coluna `country`:** Esta variável possui apenas 1 valor único (`Espana`), ou seja, variância zero. Não agrega informação estatística e pode ser desconsiderada nas análises.

---


## Análise Bivariada

### Relação entre variáveis quantitativas

In [ ]:
# @title Comportamento par a par

vars_quant = ['rating', 'price', 'num_reviews', 'body', 'acidity']
combinacoes = list(itertools.combinations(vars_quant, 2))

fig, axes = plt.subplots(figsize=(16, 30), ncols=2, nrows=5)
axes = axes.flatten()

for i, (v1, v2) in enumerate(combinacoes):

    corr = df[[v1, v2]].corr().iloc[0, 1]


    if 'body' in [v1, v2] or 'rating' in [v1, v2]:
        x_var, y_var = (v1, v2) if df[v1].nunique() < df[v2].nunique() else (v2, v1)
        sns.boxplot(data=df, x=x_var, y=y_var, ax=axes[i],
                   palette="Blues_d", hue=x_var, legend=False, showfliers=False)
        axes[i].set_title(f"Distribuição de {y_var} por {x_var}\n(r = {corr:.3f})",
                         fontsize=12, fontweight='bold')


    else:
        sns.scatterplot(data=df, x=v2, y=v1, ax=axes[i], alpha=0.3, color='#2b5b84')
        axes[i].set_xlim(0, 2000)
        axes[i].set_ylim(0, 500)
        axes[i].set_title(f"Relação: {v2} vs {v1}\n(r = {corr:.3f})",
                         fontsize=12, fontweight='bold')

    sns.despine(ax=axes[i])

plt.tight_layout()
plt.show()

---

### Análise do Comportamento Par a Par

Nesta etapa, foram analisadas as relações entre as principais variáveis do dataset. Para evitar que valores extremos prejudicassem a visualização, os eixos de price e num_reviews foram limitados e os outliers foram ocultados nos boxplots. Desse modo, ficou mais fácil enxergar as tendências centrais e os padrões mais representativos dos dados.

#### Principais Insights

**Preço vs. Rating (r = +0.545)**

- Essa é a relação mais evidente do conjunto. Conforme o rating aumenta, o preço também sobe de forma clara, principalmente a partir da nota 4.6. Vinhos com avaliações mais altas passam a ocupar faixas de preço bem superiores.

**Rating vs. Número de Reviews (r = +0.015)**

- A relação é praticamente inexistente. Ter muitas avaliações não significa, necessariamente, ter uma nota mais alta. Popularidade e qualidade percebida não caminham juntas nesse dataset.

**Body vs. Rating (r = +0.163)**

- Apesar da correlação fraca, observa-se que vinhos com body nível 5 são os que mais frequentemente atingem as notas máximas (4.8 e 4.9).

**Preço vs. Número de Reviews (r = +0.030)**

- O preço não parece influenciar o volume de avaliações. Há tanto vinhos mais acessíveis quanto vinhos de alto valor com níveis semelhantes de engajamento.

**Body vs. Preço (r = +0.154)**

- Existe uma leve tendência de vinhos mais encorpados apresentarem preços mais altos, especialmente nos níveis 4 e 5, que concentram medianas superiores em relação aos vinhos mais leves.


---

### Correlação

In [ ]:
# @title Matriz de Correlação (Heatmap)

cols_quant = ['rating', 'price', 'num_reviews', 'body', 'acidity']
corr_matrix = df[cols_quant].corr()


plt.figure(figsize=(10, 8))


sns.heatmap(corr_matrix,
            annot=True,
            fmt=".3f",
            cmap='RdBu_r',
            center=0,
            linewidths=0.5)

plt.title('Matriz de Correlação', fontsize=15, fontweight='bold')
plt.show()

---
- Corr(price, rating) > Corr(rating, body) > Corr(price, body)
---


**Preço e Qualidade (r = 0.545)**

- É a relação mais clara do dataset. Conforme o rating aumenta, o preço também sobe de forma consistente. A partir das notas 4.7, o salto de valor fica evidente, mostrando que vinhos muito bem avaliados ocupam uma faixa de preço bem mais alta.

**Num_reviews vs. Rating (r = 0.015)**

- Praticamente não há relação. Ter muitas avaliações não significa ter uma nota maior. Isso indica que popularidade e qualidade percebida não estão necessariamente conectadas.

**Body vs. Rating (r = 0.163)**

- A correlação é fraca, mas os vinhos com body nível 5 são os que aparecem com mais frequência entre as notas mais altas. Mesmo sem uma relação forte, existe um padrão visual interessante.

**Outras Variáveis (r < 0.10)**

- Acidity e número de reviews apresentam correlação muito baixa com as demais variáveis. No geral, o que mais influencia o preço dentro deste dataset é o rating, enquanto características físicas e volume de avaliações têm impacto bem menor.
---

### Relação entre variáveis qualitativas

---
#### Tratamento de Variáveis Qualitativas
Para realizar as análises de contingência e de distribuição relativa, alguns ajustes foram necessários no tratamento dos dados.

Como a variável **year** possui muitos valores únicos, a visualização direta em gráficos de barras ficaria pouco legível. Por isso, foi feita uma conversão temporária para o tipo numérico, permitindo o agrupamento em faixas de tempo (eras). Esse agrupamento facilita a identificação de padrões entre safras mais antigas e mais recentes.

Os registros classificados como “N.V.” (não-vintage) foram desconsiderados nessa etapa específica, já que não possuem referência cronológica definida e poderiam comprometer a análise temporal.

Além disso, nas grades de subplots foram considerados apenas os principais países e tipos de vinho. Essa redução de categorias ajuda a evitar excesso de informação nos gráficos e mantém o foco nos segmentos com maior representatividade no dataset.

---

In [ ]:
# @title Contingência das variáveis qualitativas

df_clean = df[df['year'] != 'N.V.'].copy()
df_clean['year'] = pd.to_numeric(df_clean['year'])
top_countries = df_clean['country'].value_counts().nlargest(3).index
top_types = df_clean['type'].value_counts().nlargest(4).index
df_map = df_clean[df_clean['country'].isin(top_countries) & df_clean['type'].isin(top_types)].copy()
df_map['era'] = pd.cut(df_map['year'], bins=[1900, 2010, 2016, 2022],
                       labels=['Até 2010', '2011-2016', 'Pós 2016'])

vars_qual = ['country', 'type', 'era']
combinacoes = list(itertools.permutations(vars_qual, 2))


fig, axes = plt.subplots(nrows=2, ncols=3, figsize=(22, 15))
axes = axes.flatten()

for i, (v1, v2) in enumerate(combinacoes):
    plot = sns.countplot(data=df_map, x=v1, hue=v2, ax=axes[i], palette='viridis')
    axes[i].set_title(f'{v1} x {v2}', fontsize=14, fontweight='bold')
    axes[i].set_ylabel('Quantidade')


    for container in axes[i].containers:
        axes[i].bar_label(container, padding=3, fontsize=10)

    axes[i].legend(title=v2.capitalize(), bbox_to_anchor=(1.02, 1), loc='upper left')
    axes[i].tick_params(axis='x', rotation=45)
    sns.despine(ax=axes[i])

plt.suptitle('Contingência das variáveis qualitativas', fontsize=18, y=1.02, fontweight='bold')
plt.tight_layout()
plt.show()

---

- O catálogo é concentrado em vinhos da Espanha, com destaque para o tipo **Rioja Red** aparece como o mais representativo, somando 2.357 unidades. Em seguida, **Ribera Del Duero Red** ocupa a segunda posição, com 1.407 registros.

- O maior volume de rótulos está concentrado no período de 2011 a 2016, que reúne 3.283 registros. Já as safras mais antigas (até 2010) representam a menor parcela do conjunto, com 802 unidades.

- Enquanto categorias específicas como **Rioja**, **Ribera** e **Priorat** apresentam queda nas safras mais recentes, o tipo genérico **“Red”** segue o movimento oposto. O volume cresce de 309 unidades (2011–2016) para 521 no período pós-2016, indicando uma expansão desse segmento nos anos mais novos.

- Há uma redução significativa no volume de **Rioja Red** nas safras mais recentes, passando de 1.526 (2011–2016) para 281 (pós-2016).

---

### Relação entre variáveis quantitativas e qualitativas

In [ ]:
if 'era' not in df_clean.columns:
    df_clean['era'] = pd.cut(df_clean['year'], bins=[1900, 2010, 2016, 2022],
                             labels=['Até 2010', '2011-2016', 'Pós 2016'])

qual_vars_impacto = ['type', 'region', 'era']
quant_vars_impacto = ['price', 'rating', 'num_reviews']

fig, axes = plt.subplots(nrows=len(qual_vars_impacto), ncols=len(quant_vars_impacto), figsize=(22, 16))

for row, qual in enumerate(qual_vars_impacto):

    top_items = df_clean[qual].value_counts().nlargest(4).index
    df_temp = df_clean[df_clean[qual].isin(top_items)].copy()

    for col, quant in enumerate(quant_vars_impacto):

        if quant == 'rating':
            sns.histplot(data=df_temp, x=quant, hue=qual, multiple='fill',
                         ax=axes[row, col], palette='magma',
                         bins=15,
                         legend=(col == len(quant_vars_impacto) - 1))
        else:
            sns.histplot(data=df_temp, x=quant, hue=qual, multiple='fill',
                         ax=axes[row, col], palette='magma', element='poly',
                         shrink=1, legend=(col == len(quant_vars_impacto) - 1))

        if row == 0:
            axes[row, col].set_title(quant.upper(), fontsize=14, fontweight='bold')
        if col == 0:
            axes[row, col].set_ylabel(qual.upper(), fontsize=14, fontweight='bold')
        else:
            axes[row, col].set_ylabel('')

        if quant == 'price': axes[row, col].set_xlim(0, 500)
        if quant == 'num_reviews': axes[row, col].set_xlim(0, 1000)

        if col == len(quant_vars_impacto) - 1:
            sns.move_legend(axes[row, col], "upper left", bbox_to_anchor=(1, 1), title=None)

plt.suptitle('Proporção por Categoria e Métrica', fontsize=20, y=1.02, fontweight='bold')
plt.tight_layout()
plt.show()

---

#### Refinamento e Seleção de Variáveis

Nesta etapa, a seleção de variáveis foi feita priorizando clareza na visualização, variáveis como *Winery* e *Wine* foram descartadas por gerarem excesso de categorias e dificultarem a identificação de padrões gerais.

A análise de integridade mostrou que as variáveis *Body* e *Acidity* possuem 1.169 valores ausentes, o que representa aproximadamente 15% do dataset. Por esse motivo, elas foram retiradas das visualizações de impacto nesta etapa, evitando interpretações baseadas em dados incompletos. Nos testes realizados, os gráficos ficaram praticamente em branco, sem curvas ou distribuições relevantes. Portanto, o tratamento desses valores ausentes será priorizado na próxima fase da análise.


A matriz final foi construída com foco nas variáveis mais explicativas: *Type*, *Region* e *Era*.

- *Price* e *Num_reviews* foram representados por curvas de densidade, preservando sua condição contínua.
- *Rating* foi apresentado em barras discretas, respeitando seu padrão fixo de valores e evitando distorções visuais.

O resultado é uma visualização mais limpa, destacando onde se concentram valor (*Price*), qualidade (*Rating*) e popularidade (*Reviews*).

---

#### Price

- No gráfico por *Type*, observa-se que **Ribera Del Duero Red** ganha participação conforme o preço aumenta, chegando a dominar quase totalmente a faixa próxima dos 500€.

- O tipo **Rioja Red** concentra a maior parte dos vinhos nas faixas de preço mais baixas. Porém, sua participação diminui gradualmente conforme os valores sobem.

- Na análise por *Era*, os vinhos até 2010 são os que mantêm presença consistente nas faixas mais altas de preço. Já os vinhos pós-2016 ficam concentrados nas faixas inferiores, sugerindo menor valorização nas safras mais recentes.


#### Rating

- Vinhos das eras até 2010 e 2011–2016 apresentam maior presença nas notas 4.8 e 4.9. Isso sugere que avaliações máximas tendem a estar associadas a safras com mais tempo de maturação.

- Os vinhos pós-2016 aparecem com frequência nas notas intermediárias, como 4.3 e 4.4. Indicam boa qualidade técnica, mas com menor presença nas pontuações mais altas.


#### Num_Reviews

- A era 2011–2016 apresenta maior participação nas diferentes faixas de número de avaliações.

- Na análise por região, **Toro** aparece com participação bem menor em número de avaliações quando comparado com **Rioja** e **Priorato**.

---

# Análise Multivariada

In [ ]:
# @title Distribuição Conjunta

# Preparar dados: converter year e criar coluna era
df_temp_for_era = df.copy()
df_temp_for_era['year'] = pd.to_numeric(df_temp_for_era['year'], errors='coerce')
df_temp_for_era = df_temp_for_era.dropna(subset=['year'])
df_temp_for_era['era'] = pd.cut(df_temp_for_era['year'], bins=[1900, 2010, 2016, 2022],
                         labels=['Até 2010', '2011-2016', 'Pós 2016'])
df_final = df_temp_for_era[df_temp_for_era['era'].isin(['Até 2010', 'Pós 2016'])].copy()

df_final['era'] = df_final['era'].cat.remove_unused_categories()

top_regions = df_final['region'].value_counts().nlargest(3).index
df_final = df_final[df_final['region'].isin(top_regions)]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 7))

sns.violinplot(
    data=df_final, x='region', y=np.log10(df_final['price']),
    hue='era', split=True, inner="quart", palette={"Até 2010": "seagreen", "Pós 2016": "orange"},
    ax=ax1
)
ax1.set_title('Contraste de Preço (Price)')
ax1.set_ylabel('Preço (Log 10)')

sns.violinplot(
    data=df_final, x='region', y='rating',
    hue='era', split=True, inner="quart", palette={"Até 2010": "seagreen", "Pós 2016": "orange"},
    ax=ax2
)
ax2.set_title('Contraste de Qualidade (Rating)')

plt.suptitle('Análise Multivariada: Relação entre variáveis qualitativas e quantitativas', fontsize=16)
plt.tight_layout()
plt.show()

---

A decisão de filtrar os dados para as três regiões com maior relevância (*Ribera del Duero*, *Priorato* e *Rioja*) e comparar apenas as eras extremas (Até 2010 vs. Pós 2016) foi tomada para garantir melhor interpretabilidade do gráfico. Pois, nas tentativas anteriores com o dataset completo, houve excesso de informação no mesmo espaço, o que dificultava a leitura dos rótulos e a identificação de padrões claros.

Portanto, ao focar nos extremos de maturidade, foi possível isolar com mais clareza o efeito do tempo sobre preço e qualidade. Além disso, a utilização de escala logarítmica no eixo Y ajudou a suavizar o impacto dos vinhos mais caros, permitindo visualizar melhor a distribuição e a densidade dos rótulos nas diferentes faixas de valor.

---

- Em todas as regiões analisadas, os vinhos da era *Até 2010* aparecem em faixas de preço claramente superiores aos da era *Pós 2016*. Em **Ribera del Duero**, por exemplo, a mediana dos vinhos antigos ultrapassa até mesmo o limite superior da maior parte dos vinhos mais novos na mesma região.

- No comparativo de qualidade, apenas os vinhos *Até 2010* conseguem alcançar e manter presença consistente nas notas mais altas (acima de 4.7). Já os vinhos *Pós 2016* apresentam uma base sólida, em torno de 4.2, mas raramente atingem o topo da escala de avaliação.

- **Rioja** mostra um padrão um pouco diferente, enquanto **Ribera del Duero** e **Priorato** apresentam um salto mais evidente de qualidade entre as eras, em **Rioja** as notas médias entre vinhos novos e antigos são mais próximas. Ainda assim, a valorização em preço dos rótulos mais antigos permanece.

- A separação visual consistente entre as eras nos gráficos reforça que tempo de maturação e região são fatores centrais na formação de preço.

---


## Mapa de Calor e Dispersão

In [ ]:
df_quant = df.copy()
df_quant['price_log'] = np.log10(df_quant['price'])

corr = df_quant[['price_log', 'rating', 'num_reviews']].corr()

fig, axes = plt.subplots(figsize=(18, 6), ncols=2)

sns.heatmap(corr, annot=True, cmap='coolwarm', fmt=".2f", ax=axes[0])
axes[0].set_title("Correlação: Price vs Rating vs Reviews")

sns.scatterplot(
    data=df_quant,
    x='rating',
    y='price_log',
    size='num_reviews',
    hue='rating',
    palette='viridis',
    sizes=(20, 500),
    alpha=0.6,
    ax=axes[1]
)
axes[1].set_title("Dispersão: Rating vs Preço (Tamanho = Reviews)")
axes[1].set_ylabel("Preço (Log 10)")

plt.tight_layout()
plt.show()

O mapa de calor mostra uma correlação positiva de 0.64 entre *rating* e *price_log*, o que indica uma relação relativamente forte entre as duas variáveis.

Na prática, isso sugere que a pontuação do vinho explica uma parte relevante do seu valor de mercado. À medida que o *rating* aumenta, o preço tende a acompanhar esse movimento de forma consistente.
